# 05 — Curadoria ICTV ou auditoria da taxonomia existente (M01)

Escolha `mode: curate` para reconstruir a partir da planilha MSL41.v1 real e do CSV original. A ausência da planilha em modo empírico interrompe a execução. `mode: audit_existing` somente audita o Parquet selecionado; não afirma sua reconstrução e nunca altera a referência. Genus/Order ausentes ou placeholders impedem publicar nova taxonomia válida para 02/06. Labels divergentes são reportados, não corrigidos.

A taxonomia identifica táxons, não os genomas de origem dos fragmentos. Sem accession por fragmento, a proveniência permanece desconhecida. `SMOKE_TEST=1` deve ser escolhido explicitamente para a demonstração sintética; sua saída não constitui evidência empírica.

## Entradas e namespace M01

Configure `M01_CONFIG` com um YAML (modelo em `configs/reproduction_m01.yaml`) ou `M01_RUN_PLAN` com o plano M00 selecionado. O plano fixa entradas e destino; overrides conflitantes são recusados. Sem plano, cada execução reserva novo `run_id`. `DATA_ROOT`, `ORIGINAL_MAPPING_PATH`, `TAXONOMY_PATH` e os overrides descritos em `docs/m01_data_taxonomy_split.md` são aceitos sem plano. Saídas históricas permanecem somente leitura.

O modo padrão é empírico, inclusive se faltarem dados. Testes locais usam `REMOTE_SERVER=False SMOKE_TEST=1` com fixtures explícitas. Manifesto COMPLETE significa análise concluída, não integridade taxonômica nem independência genômica. Examine o relatório e `hierarchy_ready` antes de usar a hierarquia.


In [1]:
import os
import sys
from pathlib import Path

import os


current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        PROJECT_ROOT = r
        break
else:
    raise FileNotFoundError("Execute dentro do repositório cnn_metagenome")

from viral_cnn.m01_workflow import run_m01_notebook


In [3]:

os.environ["M01_MODE"] = "derive_partial_existing"
os.environ["TAXONOMY_PATH"] = (
    "/home/mlima/workspace/cnn_kraken/data_taxonomy_parquet/"
    "main_taxonomy_hybrid_ictv_ncbi.parquet"
)
os.environ["ORIGINAL_MAPPING_PATH"] = (
    "/home/mlima/workspace/dataset_kraken/species_name_ids.csv"
)

print("Modo selecionado:", os.environ["M01_MODE"])

if mode == "curate":
    os.environ.setdefault("ICTV_MASTER_PATH", "/home/mlima/workspace/cnn_kraken/ICTV_Master_Species_List_2025_MSL41.v1.xlsx")

manifest = run_m01_notebook("short/05", PROJECT_ROOT)
print("Run:", manifest["run_id"])
print("Saída:", manifest["output_dir"])
print("Status:", manifest["status"])

Modo selecionado: derive_partial_existing


[LOG_PATH] Loading M01 selected original_mapping from: /home/mlima/workspace/dataset_kraken/species_name_ids.csv
[LOG_PATH] Loading M01 selected taxonomy_path from: /home/mlima/workspace/cnn_kraken/data_taxonomy_parquet/main_taxonomy_hybrid_ictv_ncbi.parquet
[LOG_PATH] Loading M01 selected ictv_master from: /home/mlima/workspace/cnn_kraken/ICTV_Master_Species_List_2025_MSL41.v1.xlsx
[LOG_PATH] Loading M01 table from: /home/mlima/workspace/dataset_kraken/species_name_ids.csv
[LOG_PATH] Loading M01 table from: /home/mlima/workspace/cnn_kraken/data_taxonomy_parquet/main_taxonomy_hybrid_ictv_ncbi.parquet


Run: m01-a5e1773b35ab421a
Saída: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/m01-a5e1773b35ab421a/short/05
Status: COMPLETE


In [4]:
from pathlib import Path
import pandas as pd
from viral_cnn.taxonomy.curation import validate_taxonomy_identity

output_dir = Path(manifest["output_dir"])
partial_parquet = output_dir / "main_taxonomy_hybrid_PARTIAL.parquet"
if partial_parquet.exists():
    hybrid = pd.read_parquet(partial_parquet)
else:
    tax_path = manifest["configuration"].get(
        "taxonomy_path",
        "/home/mlima/workspace/cnn_kraken/data_taxonomy_parquet/main_taxonomy_hybrid_ictv_ncbi.parquet",
    )
    hybrid = pd.read_parquet(tax_path)

orig_path = manifest["configuration"].get(
    "original_mapping",
    "/home/mlima/workspace/dataset_kraken/species_name_ids.csv",
)
original = pd.read_csv(orig_path)

audit = validate_taxonomy_identity(hybrid, original)
print("Integridade:", audit["mapping_integrity_ok"], audit["mapping_errors"])
print("Hierarquia pronta:", audit["hierarchy_ready"])

for rank in ("Genus", "Order"):
    missing_ids = {label + 1 for label in audit["unresolved_ranks"][rank]}
    missing = hybrid[hybrid["id"].isin(missing_ids)]
    print(f"\n{rank}: {len(missing)} classes sem rank")
    if "taxonomy_source" in missing:
        print(missing["taxonomy_source"].value_counts(dropna=False))
    print(missing[["id", "species", rank]].head(10).to_string(index=False))

Integridade: True []
Hierarquia pronta: False

Genus: 5937 classes sem rank
taxonomy_source
NCBI Fallback    5937
Name: count, dtype: int64
 id                            species Genus
 35    Nigrospora oryzae fusarivirus 1  <NA>
 36 Sodiomyces alkalinus fusarivirus 1  <NA>
 37      Heterobasidion partitivirus P  <NA>
 38       Atkinsonella hypoxylon virus  <NA>
 39      Heterobasidion partitivirus 8  <NA>
 40     Crimson clover cryptic virus 2  <NA>
 41    Ceratocystis resinifera virus 1  <NA>
 42        Hop trefoil cryptic virus 2  <NA>
 43             Cannabis cryptic virus  <NA>
 44      Heterobasidion partitivirus 7  <NA>

Order: 9100 classes sem rank
taxonomy_source
NCBI Fallback    5937
ICTV Master      3163
Name: count, dtype: int64
 id                            species Order
 35    Nigrospora oryzae fusarivirus 1  <NA>
 36 Sodiomyces alkalinus fusarivirus 1  <NA>
 37      Heterobasidion partitivirus P  <NA>
 38       Atkinsonella hypoxylon virus  <NA>
 39      Heterobasidion 

In [5]:
checked = hybrid.copy()
if "model_label" not in checked.columns:
    checked["model_label"] = checked["id"].astype("int64") - 1

report = validate_taxonomy_identity(checked, original)
print(report["mapping_integrity_ok"], report["mapping_errors"])
print({rank: len(ids) for rank, ids in report["unresolved_ranks"].items()})

True []
{'Genus': 5937, 'Order': 9100}


## Evidência e próximo passo

O manifesto registra caminhos, hashes ou identidade limitada explícita, configuração, versões e origem sintética/observada. Consulte `output_dir` abaixo; não procure a execução mais recente por data. A taxa de overlap usa o teste integral como denominador; piloto não estima prevalência. A ausência de duplicata exata não prova independência genômica. Estes resultados não são o benchmark C03.


In [6]:
import json
print(json.dumps(manifest["metrics_summary"], indent=2, ensure_ascii=False))


{
  "mapping_integrity_ok": true,
  "mapping_errors": [],
  "unresolved_ranks": {
    "Genus": [
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48,
      49,
      50,
      51,
      52,
      53,
      54,
      55,
      56,
      57,
      58,
      59,
      60,
      61,
      62,
      63,
      64,
      65,
      66,
      67,
      68,
      69,
      70,
      71,
      72,
      73,
      74,
      75,
      76,
      77,
      78,
      79,
      80,
      81,
      82,
      83,
      84,
      85,
      86,
      87,
      88,
      89,
      90,
      91,
      92,
      93,
      94,
      95,
      96,
      97,
      98,
      99,
      100,
      101,
      102,
      103,
      104,
      105,
      106,
      107,
      108,
      109,
      110,
      111,
      136,
      137,
      138,
      139,
      140,
      141,
      142,
      143,
      151,
      152,
 